# Dia 11 — Expressões regulares

**🃏 Cartas:** 11 · 32 · 53 &nbsp;·&nbsp; **📘 Original:** *Day 11. Regular expressions*
&nbsp;·&nbsp; [repositório](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias) &nbsp;·&nbsp; [curso original do cipher387](https://github.com/cipher387/python-for-OSINT-21-days)

---

## 📖 Capítulo 11 — A conversa

Ontem a vitrine entregou um e-mail esquecido num comentário. Hoje a Iara chega com algo mais
quente: uma vítima exportou a conversa inteira com o "Kau" pelo WhatsApp e mandou o `.txt` para a
delegacia.

— São só catorze linhas — você diz, aliviado.

— Hoje. — Ela sorri. — Semana que vem serão catorze mil, de vinte vítimas diferentes. Ninguém lê isso
com marca-texto. Você vai ensinar o computador a reconhecer **formatos**: CPF tem cara de CPF, chave PIX
aleatória tem cara de UUID, telefone tem DDD.

Ela escreve no quadro `\d{3}\.\d{3}\.\d{3}-\d{2}` e espera.

— Parece xingamento em código, eu sei. Mas lê comigo: três dígitos, ponto, três dígitos, ponto, três
dígitos, hífen, dois dígitos.

Você roda o primeiro teste e a tela se enche de achados: três chaves PIX, dois CNPJs, dois e-mails.
A Iara aponta para os CNPJs.

— Agora a pegadinha. A regex diz que **parece** um CNPJ. Quem diz se **é** um CNPJ é o dígito
verificador. Lembra do Dia 2? Junta as duas coisas e me diz: o "CNPJ do sócio" existe?

In [ ]:
#@title ⚙️ Preparar o ambiente (rode esta célula primeiro)
# No Colab: baixa o repositório e entra na pasta do dia.
# No Jupyter local: o notebook já está na pasta do dia, então não faz nada.
import os, subprocess, sys
from pathlib import Path

REPO = "https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias"
if "google.colab" in sys.modules:
    destino = "/content/python-para-osint-21-dias"
    if not os.path.isdir(destino):
        subprocess.run(["git", "clone", "--depth", "1", REPO, destino], check=True)
    os.chdir(f"{destino}/dia-11-regex")

PASTA_NB = str(Path.cwd())          # usada pelas células para voltar sempre à pasta certa
print("📂 Pasta do dia:", PASTA_NB)
print("🐍 Python", sys.version.split()[0])

## 🎯 Objetivos

- Rodar o exemplo original que extrai e-mails de uma página com `re.findall`.
- Ler uma regex pedaço por pedaço: `\d`, `{n}`, `?`, `+`, `[ ]`, `( )`, `|`.
- Escrever regex **comentadas** com `re.VERBOSE` para CPF, CNPJ (inclusive alfanumérico), telefone,
  CEP, placa e chave PIX.
- Extrair entidades da conversa do caso e saber em que linha e de quem veio cada uma.
- Combinar regex + dígito verificador para eliminar falsos positivos.

## 🧠 Conceito em 2 minutos

Uma **expressão regular** (regex) é um "molde" de texto. Em vez de procurar a palavra exata
`31988880101`, você procura **qualquer coisa com o formato** de um telefone.

| Peça | Significa | Exemplo | Casa com |
|---|---|---|---|
| `\d` | um dígito | `\d\d` | `31` |
| `{n}` | repetir n vezes | `\d{5}` | `30130` |
| `?` | opcional | `-?` | `-` ou nada |
| `+` | uma ou mais vezes | `[a-z]+` | `kau` |
| `[ ]` | um dos caracteres | `[.-]` | `.` ou `-` |
| `\b` | borda de palavra | `\bBRA` | `BRA` no começo de palavra |
| `(?P<nome>…)` | grupo com nome | `(?P<ddd>\d{2})` | guarda `31` em `m["ddd"]` |

No Python: `re.findall` devolve a lista de achados; `re.finditer` devolve também **onde** cada um
está; `re.fullmatch` exige que o texto **inteiro** case (ótimo para validar um campo).

**Regra de ouro:** regex reconhece **formato**, não **verdade**. `111.111.111-11` tem formato de
CPF, mas não é válido. Por isso, em OSINT, usamos a regex para achar **candidatos** e depois um
teste extra (dígito verificador, consulta a uma base) para confirmar.

**CNPJ alfanumérico:** desde 31 de julho de 2026 a Receita Federal emite, gradualmente, CNPJs com letras
nas 12 primeiras posições (ex.: `12.ABC.345/01DE-35`). Os dois dígitos verificadores continuam
numéricos, e o cálculo é o mesmo, com cada caractere valendo `ord(c) − 48`. Regex antiga que só aceita
`\d` vai deixar esses CNPJs passarem despercebidos.

## 🥋 Nível 1 — Exercício original (cipher387)

| Arquivo | O que faz |
|---|---|
| [`original/extract_emails.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-11-regex/original/extract_emails.py) | Baixa uma página da CleanTalk e lista tudo que parece e-mail com `re.findall` |

```bash
pip install requests
cd dia-11-regex
python original/extract_emails.py
```

> 💡 Repare no comentário `# 💬` sobre `[a-zA-Z0-9-_.]`: o trecho `9-_` vira um **intervalo** e aceita
> mais caracteres do que parece. Coloque o hífen no fim dos colchetes (`[a-zA-Z0-9_.-]`) e compare.
> O autor também recomenda praticar regex na busca do Google Docs/Planilhas antes de ir para o código.

In [ ]:
#@title 📦 Instalar as bibliotecas do exercício original
%pip install -q requests

### `original/extract_emails.py` — Extrair e-mails de uma página com expressão regular

In [ ]:
%cd -q {PASTA_NB}/original
# ──────────────────────────────────────────────────────────────────────────
# Título: Extrair e-mails de uma página com expressão regular
# Código ORIGINAL de cipher387 (Cyb_detective), "Python for OSINT. 21-day course"
# Fonte: https://github.com/cipher387/python-for-OSINT-21-days/blob/main/Day_11/extract_emails.py
# Licença: MIT © 2023 Cyb_detective. O código está intacto; só foram
# acrescentados comentários em pt-BR (linhas que começam com "# 💬").
# Requer: pip install requests
# ──────────────────────────────────────────────────────────────────────────

# 💬 requests baixa a página; re (nativo) procura padrões de texto.
import requests

import re




# 💬 Página pública da CleanTalk sobre um e-mail em lista negra (alvo de exemplo do autor).
url = "https://cleantalk.org/blacklists/ivanov@gmail.com"

# 💬 .text devolve o HTML como string, pronto para a regex.
html = requests.get(url).text

# 💬 A regex lê-se assim: [a-zA-Z0-9-_.]+  → um ou mais caracteres de "nome";
# 💬 @ → a arroba; e de novo [a-zA-Z0-9-_.]+ → o "domínio".
# 💬 Atenção: dentro de [ ], o trecho "9-_" vira um INTERVALO (do '9' ao '_'), que inclui
# 💬 sem querer : ; < = > ? @ [ \ ] ^ e letras maiúsculas. Funciona, mas é mais largo do que parece.
# 💬 findall devolve uma LISTA com todos os trechos que casaram com o padrão.
result = re.findall("[a-zA-Z0-9-_.]+@[a-zA-Z0-9-_.]+", html)

# 💬 Imprime a lista (pode ter repetidos; no Dia 13 você aprende a tirá-los).
print(result)

## 🇧🇷 Nível 2 — Camada Brasil

Tudo com a **biblioteca padrão** e 100% offline.

| Arquivo | O que faz |
|---|---|
| [`br/01_padroes_br.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-11-regex/br/01_padroes_br.py) | Regex comentadas de CPF, CNPJ (numérico e alfanumérico), telefone, CEP, placas, PIX e e-mail, com bateria de testes |
| [`br/02_extrai_conversa.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-11-regex/br/02_extrai_conversa.py) | Aplica tudo em `caso/dados/conversa_whatsapp.txt`: tabela entidade → ocorrências, linha e autor |
| [`br/03_regex_mais_validacao.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-11-regex/br/03_regex_mais_validacao.py) | Regex encontra, dígito verificador confirma: os CNPJs da conversa são inválidos |

```bash
cd dia-11-regex
python br/01_padroes_br.py
python br/02_extrai_conversa.py
python br/03_regex_mais_validacao.py
```

**O que você deve ver no caso:** 3 chaves PIX diferentes (a da vitrine, a "do frete" e a "do sócio"),
2 e-mails, o telefone do vendedor `+55 31 9 8888-0101` e **dois CNPJs inválidos**: o do rodapé
(`48.217.093/0001-30`) e o "do sócio" (`31.560.427/0001-71`).

> 💡 Os campos com formulário à direita (`#@param`) podem ser editados sem mexer no código. Para testar sem internet, rode antes: `os.environ['OSINT_OFFLINE'] = '1'`.

### `br/01_padroes_br.py` — Regex à brasileira: CPF, CNPJ, telefone, CEP, placa, chave PIX e e-mail.

In [ ]:
%cd -q {PASTA_NB}
"""
Regex à brasileira: CPF, CNPJ, telefone, CEP, placa, chave PIX e e-mail.

Carta 32 (rank A) — Jutsu do Padrão Brasileiro

O que você aprende:
  • a ler uma expressão regular pedaço por pedaço (\\d, {n}, ?, [ ], ( ));
  • a escrever regex comentadas com re.VERBOSE (espaços e # viram explicação);
  • padrões dos documentos e identificadores mais comuns no Brasil;
  • a testar cada padrão com exemplos que DEVEM e que NÃO DEVEM casar.

Todos os exemplos são inventados. Como rodar (dentro da pasta dia-11-regex):
    python br/01_padroes_br.py
"""

import re

# ── Mini-dicionário de regex (cole na parede) ─────────────────────────────
#   \d       um dígito (0-9)              [A-Z]   uma letra maiúscula
#   {3}      exatamente 3 vezes           {2,5}   de 2 a 5 vezes
#   ?        "pode ter ou não" (0 ou 1)   +       uma ou mais vezes
#   [.-]     um ponto OU um hífen         \.      um ponto de verdade
#   (?: )    grupo sem captura            |       "ou"
#   \b       borda de palavra: impede casar no meio de um número maior
#   (?<!\d)  "não pode ter dígito antes"  (?!\d)  "não pode ter dígito depois"

PADROES = {
    # CPF: 000.000.000-00 ou 00000000000. Pontos e hífen são opcionais.
    "CPF": re.compile(r"""
        (?<!\d)          # não começa no meio de outro número
        \d{3} \.? \d{3} \.? \d{3}   # três blocos de 3 dígitos, ponto opcional
        -? \d{2}         # hífen opcional + 2 dígitos verificadores
        (?!\d)           # e não continua com mais dígitos
    """, re.VERBOSE),

    # CNPJ numérico: 00.000.000/0000-00 (ou só os 14 dígitos).
    "CNPJ": re.compile(r"""
        (?<!\d)
        \d{2} \.? \d{3} \.? \d{3}   # raiz: 8 dígitos
        /? \d{4}                    # barra opcional + 4 dígitos da filial (0001 = matriz)
        -? \d{2}                    # 2 dígitos verificadores
        (?!\d)
    """, re.VERBOSE),

    # CNPJ alfanumérico (novo formato da Receita a partir de julho/2026):
    # as 12 primeiras posições podem ter LETRAS maiúsculas; os 2 verificadores seguem numéricos.
    "CNPJ alfanumérico": re.compile(r"""
        (?<![A-Z0-9])
        [A-Z0-9]{2} \.? [A-Z0-9]{3} \.? [A-Z0-9]{3}
        /? [A-Z0-9]{4}
        -? \d{2}
        (?![A-Z0-9])
    """, re.VERBOSE),

    # Telefone: +55 opcional, DDD com ou sem parênteses, 9 opcional (celular), 4+4 dígitos.
    "Telefone": re.compile(r"""
        (?<!\d)
        (?: \+?55 [\s-]? )?          # código do país, opcional
        \(? [1-9]\d \)? [\s-]?       # DDD: dois dígitos, o primeiro nunca é 0
        (?: 9 [\s-]? )?              # nono dígito dos celulares, opcional
        \d{4} [\s-]? \d{4}           # número: 4 + 4
        (?!\d)
    """, re.VERBOSE),

    # CEP: 00000-000 ou 00000000.
    "CEP": re.compile(r"(?<!\d)\d{5}-?\d{3}(?!\d)"),

    # Placa Mercosul (ABC1D23) e antiga (ABC-1234).
    "Placa Mercosul": re.compile(r"\b[A-Z]{3}\d[A-Z]\d{2}\b"),
    "Placa antiga": re.compile(r"\b[A-Z]{3}-?\d{4}\b"),

    # Chave PIX aleatória: um UUID (8-4-4-4-12 caracteres hexadecimais).
    "Chave PIX aleatória": re.compile(r"\b[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}\b", re.IGNORECASE),

    # E-mail: versão mais "certinha" que a do original (o hífen fica no fim dos colchetes).
    "E-mail": re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"),
}

# ── Bateria de testes: (texto, deveria casar?) ────────────────────────────
TESTES = {
    "CPF": [("123.456.789-09", True), ("12345678909", True), ("123.456.789-0", False), ("1234567890912", False)],
    "CNPJ": [("12.345.678/0001-95", True), ("12345678000195", True), ("12.345.678/0001", False)],
    "CNPJ alfanumérico": [("12.ABC.345/01DE-35", True), ("12ABC34501DE35", True), ("12.ABC.345/01DE-3X", False)],
    "Telefone": [("(31) 98888-0101", True), ("+55 31 9 8888-0101", True), ("3133334444", True), ("98888", False)],
    "CEP": [("30130-010", True), ("30130010", True), ("3013-0010", False)],
    "Placa Mercosul": [("BRA2E19", True), ("BRA-2019", False)],
    "Placa antiga": [("BRA-2019", True), ("BRA2019", True), ("BRA2E19", False)],
    "Chave PIX aleatória": [("123e4567-e89b-12d3-a456-426614174000", True), ("123e4567-e89b-12d3-a456", False)],
    "E-mail": [("fulano.de_tal@exemplo.com.br", True), ("fulano@", False), ("@exemplo.com", False)],
}

# ── Rodar a bateria e mostrar o resultado ─────────────────────────────────
erros = 0
for nome, padrao in PADROES.items():
    print(f"\n🔎 {nome}")
    for texto, esperado in TESTES[nome]:
        # fullmatch exige que o TEXTO INTEIRO case (search aceitaria um pedaço).
        casou = padrao.fullmatch(texto) is not None
        ok = casou == esperado
        erros += not ok
        print(f"   {'✅' if ok else '❌'} {texto:<40} casou={casou}  (esperado={esperado})")

print(f"\n{'🎉 Todos os testes passaram!' if erros == 0 else f'⚠️ {erros} teste(s) falharam.'}")

# ── Regex acha FORMATO, não verdade ───────────────────────────────────────
print("\n🧠 Lembre-se: '111.111.111-11' casa com a regex de CPF, mas não é um CPF válido.")
print("   A regex encontra CANDIDATOS. Quem confirma é o dígito verificador (script 03).")

### `br/02_extrai_conversa.py` — A conversa: extrair entidades do export do WhatsApp com regex.

In [ ]:
%cd -q {PASTA_NB}
"""
A conversa: extrair entidades do export do WhatsApp com regex.

Carta 32 (rank A) — Jutsu do Padrão Brasileiro (aplicação no caso)

O que você aprende:
  • a aplicar várias regex no mesmo texto e montar uma tabela entidade → ocorrências;
  • a usar finditer para saber EM QUE LINHA cada coisa apareceu;
  • a "mascarar" o que já foi achado para um padrão não roubar pedaço de outro
    (ex.: dígitos dentro de uma chave PIX parecendo telefone);
  • a separar data, hora, autor e mensagem de cada linha com grupos nomeados.

Este script é 100% offline: lê caso/dados/conversa_whatsapp.txt (fictício).

Como rodar (dentro da pasta dia-11-regex):
    python br/02_extrai_conversa.py
"""

import csv
import re
from pathlib import Path

ARQUIVO = "conversa_whatsapp.txt"  #@param {type:"string"}

PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
PASTA_CASO = PASTA_DIA.parent / "caso" / "dados"

# Ordem importa: os mais "compridos e específicos" primeiro (PIX, e-mail, CNPJ),
# depois os que poderiam casar com pedaços deles (CPF, telefone, CEP).
PADROES = {
    "Chave PIX aleatória": r"\b[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}\b",
    "E-mail": r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}",
    "CNPJ": r"(?<!\d)\d{2}\.?\d{3}\.?\d{3}/?\d{4}-?\d{2}(?!\d)",
    "CPF": r"(?<!\d)\d{3}\.?\d{3}\.?\d{3}-?\d{2}(?!\d)",
    "Telefone": r"(?<!\d)(?:\+?55[\s-]?)?\(?[1-9]\d\)?[\s-]?(?:9[\s-]?)?\d{4}[\s-]?\d{4}(?!\d)",
    "CEP": r"(?<!\d)\d{5}-\d{3}(?!\d)",
    "Valor em R$": r"R\$\s?\d{1,3}(?:\.\d{3})*(?:,\d{2})?",
}

# Linha do export: "[28/11/2025 21:14] Kau Ofertas: mensagem"
# (?P<nome>...) é um GRUPO NOMEADO: depois pegamos com m["nome"].
LINHA = re.compile(r"^\[(?P<data>\d{2}/\d{2}/\d{4}) (?P<hora>\d{2}:\d{2})\] (?P<autor>[^:]+): (?P<msg>.*)$")

texto = (PASTA_CASO / ARQUIVO).read_text(encoding="utf-8")

achados = {nome: [] for nome in PADROES}      # entidade → lista de (valor, nº da linha, autor)
for numero, linha in enumerate(texto.splitlines(), start=1):
    m = LINHA.match(linha)
    if not m:
        continue                               # linha fora do padrão (ex.: mensagem com quebra)
    msg = m["msg"]
    for nome, padrao in PADROES.items():
        for achado in re.finditer(padrao, msg):
            achados[nome].append((achado.group(), numero, m["autor"]))
        # Mascarar: troca o que foi achado por "#" do mesmo tamanho. Assim os
        # próximos padrões não enxergam mais esses caracteres.
        msg = re.sub(padrao, lambda a: "#" * len(a.group()), msg)

# ── Tabela entidade → ocorrências ─────────────────────────────────────────
print(f"{'Entidade':<22} {'Qtde':>4}  Valores (linha · autor)")
print("─" * 78)
for nome, lista in achados.items():
    if not lista:
        continue
    print(f"{nome:<22} {len(lista):>4}  {lista[0][0]}  (l.{lista[0][1]} · {lista[0][2]})")
    for valor, numero, autor in lista[1:]:
        print(f"{'':<28}{valor}  (l.{numero} · {autor})")

# ── O que o VENDEDOR revelou (só mensagens dele) ──────────────────────────
print("\n🎯 Identificadores ditos pelo vendedor:")
for nome, lista in achados.items():
    do_vendedor = sorted({valor for valor, _, autor in lista if autor.startswith("Kau")})
    if do_vendedor:
        print(f"   {nome:<22} {', '.join(do_vendedor)}")

# ── Salvar para os próximos dias ──────────────────────────────────────────
saidas = PASTA_DIA / "saidas"
saidas.mkdir(exist_ok=True)
with open(saidas / "entidades_conversa.csv", "w", newline="", encoding="utf-8-sig") as f:
    escritor = csv.writer(f, delimiter=";")
    escritor.writerow(["entidade", "valor", "linha", "autor"])
    for nome, lista in achados.items():
        for valor, numero, autor in lista:
            escritor.writerow([nome, valor, numero, autor])
print("\n💾 saidas/entidades_conversa.csv gravado.")

### `br/03_regex_mais_validacao.py` — Regex acha, dígito verificador confirma: menos falso positivo.

In [ ]:
%cd -q {PASTA_NB}
"""
Regex acha, dígito verificador confirma: menos falso positivo.

Carta 32 (rank A) — Jutsu do Padrão Brasileiro (parte 3)

O que você aprende:
  • por que "casar com a regex" não é o mesmo que "ser um documento válido";
  • a juntar a regex do Dia 11 com o cálculo de dígito verificador do Dia 2;
  • a validar CNPJ numérico E alfanumérico com a mesma função (valor = ord(c) − 48);
  • a classificar cada achado em ✅ válido ou 🚩 inválido.

Este script é 100% offline: lê caso/dados/conversa_whatsapp.txt (fictício).

Como rodar (dentro da pasta dia-11-regex):
    python br/03_regex_mais_validacao.py
"""

import re
from pathlib import Path

ARQUIVO = "conversa_whatsapp.txt"  #@param {type:"string"}

PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
PASTA_CASO = PASTA_DIA.parent / "caso" / "dados"

RE_CPF = re.compile(r"(?<!\d)\d{3}\.?\d{3}\.?\d{3}-?\d{2}(?!\d)")
# Aceita CNPJ numérico e alfanumérico (letras só nas 12 primeiras posições).
RE_CNPJ = re.compile(r"(?<![A-Z0-9])[A-Z0-9]{2}\.?[A-Z0-9]{3}\.?[A-Z0-9]{3}/?[A-Z0-9]{4}-?\d{2}(?![A-Z0-9])")


def so_caracteres(texto):
    """Tira pontos, barras e hífens: '48.217.093/0001-30' → '48217093000130'."""
    return re.sub(r"[^0-9A-Z]", "", texto.upper())


def cpf_valido(cpf):
    """Confere os 2 dígitos verificadores do CPF (mesmo algoritmo do Dia 2)."""
    n = [int(c) for c in so_caracteres(cpf)]
    if len(n) != 11 or len(set(n)) == 1:        # 111.111.111-11 passa na conta, mas é inválido
        return False
    for tamanho in (9, 10):                      # 1º DV usa 9 dígitos; 2º DV usa 10
        soma = sum(d * peso for d, peso in zip(n[:tamanho], range(tamanho + 1, 1, -1)))
        dv = (soma * 10) % 11 % 10               # resto 10 vira 0
        if dv != n[tamanho]:
            return False
    return True


def cnpj_valido(cnpj):
    """
    Confere os DVs do CNPJ. Funciona para o formato numérico e o alfanumérico:
    cada caractere vale ord(c) - 48 ('0'→0 … '9'→9, 'A'→17, 'B'→18 …). Pesos iguais aos de sempre.
    """
    c = so_caracteres(cnpj)
    if len(c) != 14 or not c[-2:].isdigit() or len(set(c)) == 1:
        return False
    valores = [ord(ch) - 48 for ch in c]
    pesos = [5, 4, 3, 2, 9, 8, 7, 6, 5, 4, 3, 2]
    for posicao in (12, 13):                     # 1º DV olha 12 caracteres; 2º olha 13
        p = pesos if posicao == 12 else [6] + pesos
        resto = sum(v * peso for v, peso in zip(valores[:posicao], p)) % 11
        dv = 0 if resto < 2 else 11 - resto
        if dv != valores[posicao]:
            return False
    return True


# ── 1. Teste rápido das funções com exemplos conhecidos ───────────────────
print("🧪 Autoteste:")
for exemplo, func, esperado in [
    ("00.000.000/0001-91", cnpj_valido, True),     # Banco do Brasil (CNPJ público)
    ("12.ABC.345/01DE-35", cnpj_valido, True),     # exemplo alfanumérico usado pela Receita
    ("111.111.111-11", cpf_valido, False),         # casa com a regex, mas é inválido
]:
    print(f"   {'✅' if func(exemplo) == esperado else '❌'} {exemplo:<20} válido={func(exemplo)}")

# ── 2. Aplicar na conversa ────────────────────────────────────────────────
texto = (PASTA_CASO / ARQUIVO).read_text(encoding="utf-8")

# Os CNPJs são procurados primeiro e depois "apagados" do texto, para que
# os 11 primeiros dígitos de um CNPJ não sejam confundidos com um CPF.
cnpjs = RE_CNPJ.findall(texto)
texto_sem_cnpj = RE_CNPJ.sub(" ", texto)
cpfs = RE_CPF.findall(texto_sem_cnpj)

print(f"\n🔎 Candidatos encontrados pela regex: {len(cnpjs)} CNPJ, {len(cpfs)} CPF\n")
invalidos = 0
for tipo, lista, validar in [("CNPJ", cnpjs, cnpj_valido), ("CPF", cpfs, cpf_valido)]:
    for doc in lista:
        ok = validar(doc)
        invalidos += not ok
        print(f"   {tipo:<5} {doc:<20} {'✅ dígito verificador confere' if ok else '🚩 INVÁLIDO (DV não confere)'}")

# ── 3. Leitura de analista ────────────────────────────────────────────────
print("\n🧠 Os dois CNPJs que o vendedor mandou ('pode confiar' e 'do sócio') são inválidos:")
print("   nenhuma empresa registrada pode ter esses números. Isso é pista de fraude e")
print("   também evita gastar consulta em API de CNPJ à toa.")
print("   O CPF válido é da VÍTIMA (enviado 'pra nota'): dado pessoal, trate com cuidado (LGPD).")
print(f"\n📊 Resumo: {len(cnpjs) + len(cpfs)} candidatos, {invalidos} inválidos descartados.")

## ⚖️ Ética, LGPD e OPSEC

- **Extrair em massa é tratar dado pessoal em massa.** Uma regex de CPF passada num vazamento devolve
  milhares de titulares. Aplique só em material que você tem base legal para analisar e guarde apenas o
  necessário para a finalidade.
- **Vítima também aparece na extração.** O CPF da conversa é da cliente, não do golpista. No relatório,
  separe "dados do investigado" de "dados de vítimas" e minimize (mascare) os das vítimas.
- **Falso positivo pode acusar inocente.** Um número que "parece CNPJ" pode ser um código de rastreio.
  Sempre valide (dígito verificador, fonte oficial) antes de afirmar qualquer coisa.
- **Não cole conversas reais em sites de "testar regex" online.** Teste com dados inventados (como os
  do script 01) e rode o material real só na sua máquina.

## 🎯 Quiz rápido

Escolha a resposta no formulário de cada célula e rode a célula.

In [ ]:
#@title 1. Qual regex casa com um CEP escrito como 30130-010 ou 30130010?
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) \\d{5}-\\d{3}", "B) \\d{5}-?\\d{3}", "C) \\d{8}", "D) [0-9]+-[0-9]+"]
correta = "B) \\d{5}-?\\d{3}"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "O ? torna o hífen opcional: 5 dígitos, hífen ou nada, 3 dígitos.")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

In [ ]:
#@title 2. A regex encontrou '31.560.427/0001-71' na conversa. O que isso prova?
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) Que a empresa existe", "B) Que o texto tem o FORMATO de um CNPJ; falta conferir o dígito verificador", "C) Que é o CNPJ do Banco do Brasil", "D) Nada, regex não funciona com barra"]
correta = "B) Que o texto tem o FORMATO de um CNPJ; falta conferir o dígito verificador"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "Regex reconhece formato. Neste caso o dígito verificador não confere: o número é inválido.")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

In [ ]:
#@title 3. No CNPJ alfanumérico, como uma letra entra no cálculo do dígito verificador?
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) Ela é ignorada", "B) Vale a posição no alfabeto (A=1)", "C) Vale ord(caractere) − 48 (A=17)", "D) Vale sempre zero"]
correta = "C) Vale ord(caractere) − 48 (A=17)"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "A regra da Receita usa o código ASCII menos 48: '0'..'9' continuam 0..9 e 'A' vale 17.")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

# 🏆 Desafio S — Dia 11: O elo telefônico

**Carta 53 · Rank S**

## A missão

O mesmo telefone aparece escrito de cinco jeitos: `(31) 98888-0101`, `+55 31 9 8888-0101`,
`5531988880101` (dentro do link `wa.me/`), `31988880101`... Para o computador, são cinco coisas
diferentes. A Iara quer **uma** forma canônica e um mapa de onde cada número aparece.

Escreva um script que:

1. Procure telefones em **todas** as fontes de texto do caso: `conversa_whatsapp.txt`,
   `vitrine_megaoferta.html`, `depoimentos/*.txt` e a coluna `telefone_contato` de `denuncias.csv`.
2. **Normalize** cada número para `+55` + DDD + número (ex.: `+5531988880101`).
3. Monte um dicionário `telefone → conjunto de fontes`.
4. Mostre os números que aparecem em pelo menos `MIN_FONTES` fontes (padrão: 2), do mais citado
   para o menos citado.

## Critérios de rank S

- [ ] Só biblioteca padrão (`re`, `csv`, `pathlib`, `collections`)
- [ ] Os dígitos de chaves PIX e CPFs **não** viram telefones falsos
- [ ] O número dentro de `wa.me/5531988880101` é reconhecido
- [ ] O telefone do vendedor aparece no topo, ligando vitrine, conversa e depoimentos
- [ ] Um arquivo faltando não derruba o script

## Dica

Normalizar é mais fácil do que parece: `re.sub(r"\D", "", texto)` deixa só os dígitos. Depois é contar:
10 ou 11 dígitos = DDD + número; 12 ou 13 começando com `55` = com código do país.

In [ ]:
%cd -q {PASTA_NB}
# ✍️ Escreva sua solução aqui.

In [ ]:
#@title 🔒 Gabarito (dê dois cliques aqui para ver o código) { display-mode: "form" }
%cd -q {PASTA_NB}
"""
Gabarito do Desafio S do Dia 11 — Telefones normalizados e cruzados entre fontes.

Procura telefones em TODAS as fontes de texto do caso (conversa, vitrine, depoimentos
e a coluna telefone_contato das denúncias), normaliza cada um para o formato
+55DDNNNNNNNNN e mostra quais números aparecem em mais de uma fonte.
100% offline: só caso/dados (fictício).
"""

import csv
import re
from collections import defaultdict
from pathlib import Path

MIN_FONTES = 2  #@param {type:"integer"}

PASTA_DIA = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
PASTA_CASO = PASTA_DIA.parent / "caso" / "dados"

RE_TEL = re.compile(r"(?<![\d-])(?:\+?55[\s-]?)?\(?[1-9]\d\)?[\s-]?(?:9[\s-]?)?\d{4}[\s-]?\d{4}(?![\d-])")
RE_UUID = re.compile(r"[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}", re.I)
RE_CPF = re.compile(r"\d{3}\.\d{3}\.\d{3}-\d{2}")


def normalizar(bruto):
    """'(31) 98888-0101', '+55 31 9 8888-0101', '5531988880101' → '+5531988880101'."""
    digitos = re.sub(r"\D", "", bruto)
    if len(digitos) in (12, 13) and digitos.startswith("55"):
        digitos = digitos[2:]                     # tira o código do país
    if len(digitos) not in (10, 11):
        return None                               # não parece telefone brasileiro
    return "+55" + digitos


def telefones_no_texto(texto):
    # Apaga UUIDs e CPFs antes, para seus dígitos não virarem "telefones".
    texto = RE_CPF.sub(" ", RE_UUID.sub(" ", texto))
    return [t for t in (normalizar(m) for m in RE_TEL.findall(texto)) if t]


fontes = defaultdict(set)         # telefone → conjunto de fontes onde apareceu

# 1) Arquivos de texto do caso
arquivos = [PASTA_CASO / "conversa_whatsapp.txt", PASTA_CASO / "vitrine_megaoferta.html"]
arquivos += sorted((PASTA_CASO / "depoimentos").glob("*.txt"))
for arquivo in arquivos:
    try:
        texto = arquivo.read_text(encoding="utf-8")
    except OSError as erro:
        print(f"⚠️  Pulei {arquivo.name}: {erro}")
        continue
    # A vitrine também tem o número "colado" no link wa.me/5531988880101
    texto = re.sub(r"wa\.me/(\d+)", r" +\1 ", texto)
    for tel in telefones_no_texto(texto):
        fontes[tel].add(arquivo.name)

# 2) Coluna da planilha de denúncias
with open(PASTA_CASO / "denuncias.csv", encoding="utf-8") as f:
    for linha in csv.DictReader(f, delimiter=";"):
        tel = normalizar(linha["telefone_contato"])
        if tel:
            fontes[tel].add(f"denuncias.csv ({linha['id']})")

print(f"📞 Telefones distintos encontrados: {len(fontes)}\n")
cruzados = {t: f for t, f in fontes.items() if len(f) >= MIN_FONTES}
print(f"🔗 Aparecem em {MIN_FONTES}+ fontes: {len(cruzados)}")
for tel, onde in sorted(cruzados.items(), key=lambda par: -len(par[1])):
    print(f"   {tel}  ← {len(onde)} fontes: {', '.join(sorted(onde))}")

print("\n🧠 O número do vendedor aparece na vitrine, na conversa e nos depoimentos:")
print("   é o elo que liga o site, o atendimento e as vítimas.")

## 🔎 Inovação OSINT do dia

**`iocextract`: regex prontas para indicadores de ameaça.** Em relatórios de segurança, URLs e IPs
costumam vir "desarmados" (`hxxp://exemplo[.]com`, `10[.]0[.]0[.]1`) para ninguém clicar sem querer.
A biblioteca Python `iocextract` (InQuest) já traz regex para achar URLs, IPs, e-mails e hashes nesse
formato e ainda "rearma" o texto:

```python
# pip install iocextract
import iocextract
texto = "Golpe em hxxps://megaoferta-brasil[.]example/pix"
print(list(iocextract.extract_urls(texto, refang=True)))
```

Boa companhia para os padrões brasileiros deste dia, que ela não cobre (CPF, CNPJ, PIX).

## 📚 Para ir além

- Módulo `re` (documentação oficial): <https://docs.python.org/pt-br/3/library/re.html>
- HOWTO de expressões regulares: <https://docs.python.org/pt-br/3/howto/regex.html>
- Testar regex de forma visual (use só dados inventados): <https://regex101.com/>
- Ministério da Fazenda, "Receita Federal gera o primeiro CNPJ em formato alfanumérico" (jul/2026): <https://www.gov.br/fazenda/pt-br/assuntos/noticias/2026/julho/receita-federal-gera-o-primeiro-cnpj-em-formato-alfanumerico/>
- iocextract: <https://github.com/InQuest/iocextract>

## ✅ Checklist

- [ ] Rodei o script original e corrigi o intervalo `9-_` da regex
- [ ] Todos os testes do `01_padroes_br.py` passaram
- [ ] Extraí as entidades da conversa e sei qual autor disse cada uma
- [ ] Confirmei que os dois CNPJs do vendedor são inválidos
- [ ] Fiz o desafio S
- [ ] Anotei no meu [`DIARIO.md`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/DIARIO.md)

---

🎉 Terminou? [Registre sua entrega do Dia 11](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/issues/new?template=entreguei-o-dia.yml) · Baseado em [cipher387/python-for-OSINT-21-days](https://github.com/cipher387/python-for-OSINT-21-days) (MIT) · Adaptação pt-BR: [Ridd1kulusC0d3r/python-para-osint-21-dias](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias)